# Aula 13 — BFS, DFS e árvores de busca em tubulações

**ECAA08 · Grupo 06 · Drone agrícola de pulverização**


## Perguntas de supervisão

Quais bicos são alcançáveis a partir do reservatório? Que equipamentos ficam a jusante de um bloqueio? Há outro caminho filtrado até a bomba? As buscas recebem o modo operacional e um conjunto de vértices bloqueados, sem alterar o catálogo original.


## Busca em largura e sua árvore

A BFS utiliza fila FIFO, marca cada vértice ao descobri-lo e atribui um pai. A primeira descoberta ocorre com o menor número de arestas desde a origem. A árvore de predecessores tem um arco por vértice alcançado, exceto a raiz: $|E_T|=|V_T|-1$. Ela preserva a alcançabilidade da busca, mas não todas as redundâncias da rede.

Da raiz `DRN_TQ_01`, as camadas são: tanque (0), filtros (1), bomba (2), distribuidor (3), bicos (4). Na configuração básica há apenas o filtro principal; na proposta redundante aparecem os dois filtros na camada 1. A BFS escolhe um deles para ser pai da bomba, embora ambos possam alcançá-la.

Não se interpreta o número de arestas como número de válvulas: alguns trechos não possuem válvula própria. A BFS minimiza saltos, enquanto Dijkstra minimiza a soma dos pesos.


## Busca em profundidade

O algoritmo implementado enumera **caminhos simples** por retrocesso: ao avançar por um ramo, impede repetir vértices daquele caminho; ao voltar, permite explorar outros ramos. Na extensão redundante existem dois caminhos do tanque a `DRN_BICO_A`, um por filtro. Ao bloquear `DRN_FLT_01`, resta apenas o caminho via `DRN_FLT_02`.

Uma DFS de visita única custa $O(V+E)$, assim como a BFS com listas de adjacência. A **enumeração de todos os caminhos** pode ter quantidade fatorial de resultados em grafos densos; não recebe a mesma garantia linear. Por isso a enumeração aqui é restrita à rede pequena da simulação.


## Exemplos e ligação com a Etapa 1

Bloquear a bomba elimina todos os caminhos até os bicos, apesar da redundância dos filtros. Bloquear a origem ou o destino retorna lista vazia. A identidade origem=destino produz caminho de zero arestas, desde que o nó esteja disponível.

A condição de conectividade de aplicação é $\forall b\in\mathcal B,\;\operatorname{alcancavel}(tanque,b)$, com $\mathcal B$ não vazio. Ela complementa os quantificadores da Aula 06, mas não substitui pressão, vazão e integridade de telemetria.

```mermaid
flowchart LR
    T[DRN_TQ_01] --> F1[DRN_FLT_01]
    T --> F2[DRN_FLT_02]
    F1 --> P[DRN_PMP_01]
    P --> M[DRN_MAN_01]
    M --> A[DRN_BICO_A]
    M --> B[DRN_BICO_B]
    M --> C[DRN_BICO_C]
    M --> D[DRN_BICO_D]
```

O desenho representa a árvore BFS nominal na extensão; a ligação física F2→P foi excluída da árvore, não removida da planta.

**Entregável:** BFS com árvore e camadas; DFS com enumeração; cenários de filtro bloqueado, bomba indisponível, extremos bloqueados e ausência de conexão solo–drone em voo.


## Implementação executável

Python 3.10 ou superior. Execute de cima para baixo. As definições estão incluídas para permitir abrir apenas este notebook no Jupyter ou Colab. Nenhuma conexão com equipamento real é realizada.

In [1]:
from collections import deque, Counter
from math import inf, isfinite, hypot
from heapq import heappush, heappop
from itertools import permutations
from copy import deepcopy


In [2]:
def tabela(linhas):
    if not linhas:
        return '(sem registros)'
    cols = list(linhas[0])
    larg = {c: max(len(c), *(len(str(r.get(c, ''))) for r in linhas)) for c in cols}
    fmt = lambda r: ' | '.join(str(r.get(c, '')).ljust(larg[c]) for c in cols)
    return '\n'.join([fmt(dict(zip(cols, cols))), '-+-'.join('-'*larg[c] for c in cols)] + [fmt(r) for r in linhas])


In [3]:
def matriz_texto(M, linhas, colunas):
    return tabela([{'no': r, **{c: ('inf' if M[i][j] == inf else round(M[i][j], 3))
                   for j, c in enumerate(colunas)}} for i, r in enumerate(linhas)])


In [4]:
class GrafoTubulacao:
    """Dígrafo simples; catálogo físico e disponibilidade operacional separados.

    Sensores são atributos; válvulas podem estar nas arestas. Não há suposição
    de uma válvula em cada trecho. Arestas paralelas são rejeitadas.
    """
    def __init__(self, vertices):
        self.vertices = list(vertices)
        if len(set(self.vertices)) != len(self.vertices):
            raise ValueError('Vértices duplicados')
        self.arestas = {}
        self.adj = {v: [] for v in self.vertices}

    def adicionar_tubulacao(self, id_, origem, destino, comprimento_m,
                           valvula=None, diametro_mm=8.0, modos=('voo',)):
        if origem not in self.adj or destino not in self.adj:
            raise KeyError('Extremidade não cadastrada')
        if origem == destino or id_ in self.arestas:
            raise ValueError('Laço ou ID repetido')
        if any(self.arestas[e]['destino'] == destino for e in self.adj[origem]):
            raise ValueError('Aresta paralela: use um multigrafo com IDs próprios')
        if not isfinite(comprimento_m) or comprimento_m < 0 or not isfinite(diametro_mm) or diametro_mm <= 0:
            raise ValueError('Peso finito não negativo e diâmetro positivo são exigidos')
        self.arestas[id_] = dict(origem=origem, destino=destino, peso=float(comprimento_m),
                                 valvula=valvula, diametro_mm=diametro_mm,
                                 modos=tuple(modos), disponivel=True)
        self.adj[origem].append(id_)

    def vizinhos(self, u, modo=None, bloqueados=()):
        if u not in self.adj:
            raise KeyError(u)
        if u in bloqueados:
            return
        for id_ in self.adj[u]:
            a = self.arestas[id_]
            if a['disponivel'] and (modo is None or modo in a['modos']) and a['destino'] not in bloqueados:
                yield a['destino'], a['peso'], id_

    def bloquear(self, id_):
        self.arestas[id_]['disponivel'] = False

    def matrizes(self, modo=None):
        n = len(self.vertices)
        idx = {v: i for i, v in enumerate(self.vertices)}
        ids = list(self.arestas)
        A = [[0]*n for _ in range(n)]
        W = [[0.0 if i == j else inf for j in range(n)] for i in range(n)]
        B = [[0]*len(ids) for _ in range(n)]
        for j, id_ in enumerate(ids):
            a = self.arestas[id_]; u, v = idx[a['origem']], idx[a['destino']]
            # B descreve o catálogo físico, inclusive trechos indisponíveis.
            B[u][j], B[v][j] = -1, 1
            if a['disponivel'] and (modo is None or modo in a['modos']):
                A[u][v], W[u][v] = 1, a['peso']
        return A, W, B, ids

    def graus(self, modo=None):
        A, _, _, _ = self.matrizes(modo)
        return [{'no': v, 'entrada': sum(r[i] for r in A), 'saida': sum(A[i])}
                for i, v in enumerate(self.vertices)]


In [5]:
def criar_rede(redundante=False):
    nos = ['EST_TQ_01', 'EST_PMP_01', 'EST_MAN_01', 'DRN_TQ_01',
           'DRN_FLT_01', 'DRN_PMP_01', 'DRN_MAN_01'] + ['DRN_BICO_'+s for s in 'ABCD']
    if redundante:
        nos.append('DRN_FLT_02')
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao('e01', nos[0], nos[1], 1.5, diametro_mm=20, modos=('mistura','abastecimento'))
    g.adicionar_tubulacao('e02', nos[1], nos[2], 1.0, diametro_mm=20, modos=('mistura','abastecimento'))
    g.adicionar_tubulacao('e03', nos[2], nos[0], 2.0, 'EST_XV_REC', 20, ('mistura',))
    g.adicionar_tubulacao('e04', nos[2], nos[3], 3.0, 'EST_XV_01', 15, ('abastecimento',))
    g.adicionar_tubulacao('e05', nos[3], nos[4], 0.4, 'DRN_XV_F1_IN')
    g.adicionar_tubulacao('e06', nos[4], nos[5], 0.3, 'DRN_XV_F1_OUT')
    g.adicionar_tubulacao('e07', nos[5], nos[6], 0.5)
    for i, s in enumerate('ABCD'):
        g.adicionar_tubulacao(f'e{8+i:02}', nos[6], 'DRN_BICO_'+s, 0.6+0.2*i, 'DRN_XV_01'+s)
    if redundante:
        g.adicionar_tubulacao('e12', 'DRN_TQ_01', 'DRN_FLT_02', 0.5, 'DRN_XV_F2_IN')
        g.adicionar_tubulacao('e13', 'DRN_FLT_02', 'DRN_PMP_01', 0.6, 'DRN_XV_F2_OUT')
    return g


In [6]:
def reconstruir(pred, destino):
    if destino not in pred:
        return []
    caminho, u = [], destino
    while u is not None:
        caminho.append(u); u = pred[u]
    return caminho[::-1]


In [7]:
def bfs(g, origem, destino, modo=None, bloqueados=()):
    if origem not in g.adj or destino not in g.adj:
        raise KeyError('Origem/destino inexistente')
    if origem in bloqueados or destino in bloqueados:
        return [], {}, {}
    pred, nivel, fila = {origem: None}, {origem: 0}, deque([origem])
    while fila:
        u = fila.popleft()
        for v, _, _ in g.vizinhos(u, modo, bloqueados):
            if v not in pred:
                pred[v], nivel[v] = u, nivel[u]+1
                fila.append(v)
    return reconstruir(pred, destino), pred, nivel


In [8]:
def dfs_caminhos(g, origem, destino, modo=None, bloqueados=()):
    """Enumera caminhos simples. Custo depende do número de caminhos, não é O(V+E)."""
    if origem not in g.adj or destino not in g.adj:
        raise KeyError('Origem/destino inexistente')
    if origem in bloqueados or destino in bloqueados:
        return []
    resultado = []
    def visitar(u, caminho):
        if u == destino:
            resultado.append(caminho[:]); return
        for v, _, _ in g.vizinhos(u, modo, bloqueados):
            if v not in caminho:
                visitar(v, caminho+[v])
    visitar(origem, [origem])
    return resultado


In [9]:
def dijkstra(g, origem, destino, modo=None, bloqueados=()):
    if origem not in g.adj or destino not in g.adj:
        raise KeyError('Origem/destino inexistente')
    if origem in bloqueados or destino in bloqueados:
        return inf, []
    dist, pred, fila = {origem: 0.0}, {origem: None}, [(0.0, origem)]
    while fila:
        du, u = heappop(fila)
        if du != dist[u]:
            continue
        if u == destino:
            return du, reconstruir(pred, destino)
        for v, peso, _ in g.vizinhos(u, modo, bloqueados):
            if not isfinite(peso) or peso < 0:
                raise ValueError('Dijkstra exige pesos finitos não negativos')
            candidato = du+peso
            if candidato < dist.get(v, inf):
                dist[v], pred[v] = candidato, u
                heappush(fila, (candidato, v))
    return inf, []


## Experimento 1 — BFS, camadas e árvore

In [10]:
rede = criar_rede(True)
rota,pais,camadas = bfs(rede,'DRN_TQ_01','DRN_BICO_A','voo')
print('BFS:', ' -> '.join(rota))
print(tabela([{'no':v,'pai':pais[v],'camada':camadas[v]} for v in pais]))
arvore = [(p,v) for v,p in pais.items() if p is not None]
assert len(arvore)==len(pais)-1
assert camadas['DRN_BICO_A']==4
assert all(camadas[v]==camadas[p]+1 for p,v in arvore)


BFS: DRN_TQ_01 -> DRN_FLT_01 -> DRN_PMP_01 -> DRN_MAN_01 -> DRN_BICO_A
no         | pai        | camada
-----------+------------+-------
DRN_TQ_01  | None       | 0     
DRN_FLT_01 | DRN_TQ_01  | 1     
DRN_FLT_02 | DRN_TQ_01  | 1     
DRN_PMP_01 | DRN_FLT_01 | 2     
DRN_MAN_01 | DRN_PMP_01 | 3     
DRN_BICO_A | DRN_MAN_01 | 4     
DRN_BICO_B | DRN_MAN_01 | 4     
DRN_BICO_C | DRN_MAN_01 | 4     
DRN_BICO_D | DRN_MAN_01 | 4     


## Experimento 2 — DFS e falhas de equipamentos

In [11]:
caminhos = dfs_caminhos(rede,'DRN_TQ_01','DRN_BICO_A','voo')
for r in caminhos: print('DFS:', ' -> '.join(r))
assert len(caminhos)==2
alternativas = dfs_caminhos(rede,'DRN_TQ_01','DRN_BICO_A','voo',{'DRN_FLT_01'})
assert len(alternativas)==1 and 'DRN_FLT_02' in alternativas[0]
for bloqueados in [{'DRN_PMP_01'},{'DRN_TQ_01'},{'DRN_BICO_A'}]:
    assert bfs(rede,'DRN_TQ_01','DRN_BICO_A','voo',bloqueados)[0]==[]
    assert dfs_caminhos(rede,'DRN_TQ_01','DRN_BICO_A','voo',bloqueados)==[]
assert bfs(rede,'DRN_TQ_01','DRN_TQ_01','voo')[0]==['DRN_TQ_01']
assert bfs(rede,'EST_TQ_01','DRN_TQ_01','voo')[0]==[]
print('Duas rotas; uma com filtro principal bloqueado; nenhuma com bomba bloqueada.')


DFS: DRN_TQ_01 -> DRN_FLT_01 -> DRN_PMP_01 -> DRN_MAN_01 -> DRN_BICO_A
DFS: DRN_TQ_01 -> DRN_FLT_02 -> DRN_PMP_01 -> DRN_MAN_01 -> DRN_BICO_A
Duas rotas; uma com filtro principal bloqueado; nenhuma com bomba bloqueada.
